In [1]:
import re  # noqa: F401

import pandas as pd


In [2]:
df = pd.read_csv('it_vacancies_2024.csv')

In [4]:
df.columns

Index(['source', 'vacancy_id', 'status', 'snapshot_date', 'published_at',
       'area', 'employer_id', 'employer_inn', 'employer_ogrn', 'employer_name',
       'name', 'role', 'sphere', 'salary_from', 'salary_to', 'currency',
       'gross', 'experience', 'schedule', 'employment', 'key_skills',
       'description', 'url'],
      dtype='str')

In [5]:
df["role"].unique()

<StringArray>
[                    'Программист, разработчик',
                           'Системный аналитик',
                              'Сетевой инженер',
                                     'Аналитик',
                              'Бизнес-аналитик',
    'Специалист по информационной безопасности',
                               'DevOps-инженер',
                      'Системный администратор',
                         'Технический писатель',
                                  'Тестировщик',
                            'Системный инженер',
                               'Дата-сайентист',
                 'BI-аналитик, аналитик данных',
               'Руководитель группы разработки',
                         'Продуктовый аналитик',
 'Директор по информационным технологиям (CIO)',
                'Руководитель отдела аналитики']
Length: 17, dtype: str

In [17]:
print(f"Всего вакансий: {len(df)}")

# Оставляем только нужные роли
roles_to_include = ["Программист, разработчик", "Руководитель группы разработки"]
df_filtered = df[df["role"].isin(roles_to_include)].copy()
print(f"Вакансий с нужными ролями: {len(df_filtered)}")

# Ищем только по name
text_to_search = (
    df_filtered["name"].fillna("")
).str.lower()

# Паттерны для поиска C/C++
# Ищем: c++, cpp, си++, а также c или си как отдельные слова
patterns = r"c\+\+|cpp|си\+\+|\bc\b|\bси\b"

# Паттерны для исключения ложных срабатываний
# Исключаем: c#, c-sharp, си шарп, 1c, 1с, css, c-level, scala, objective-c
patterns_exclude = r"c#|c-sharp|си шарп|1c|1с|css|c-level|scala|objective-c|java"

# 6. Итоговая маска
mask_final = (
    text_to_search.str.contains(patterns, regex=True) & 
    ~text_to_search.str.contains(patterns_exclude, regex=True)
)

df_cpp = df_filtered[mask_final].copy()
print(f"\nНайдено вакансий C/C++: {len(df_cpp)}")

# Проверка: смотрим, что попало в выборку
print("\nТоп-10 названий найденных вакансий:")
print(df_cpp["name"].value_counts().head(1).to_string())

print("\nРаспределение по ролям:")
print(df_cpp["role"].value_counts().to_string())

Всего вакансий: 229388
Вакансий с нужными ролями: 93407

Найдено вакансий C/C++: 2125

Топ-10 названий найденных вакансий:
name
Программист C/С++    92

Распределение по ролям:
role
Программист, разработчик          2051
Руководитель группы разработки      74


In [3]:
print(f"Всего вакансий: {len(df)}")

# Фильтр по ролям
roles_to_include = ["Программист, разработчик", "Руководитель группы разработки"]
df_filtered = df[df["role"].isin(roles_to_include)].copy()
print(f"Вакансий с нужными ролями: {len(df_filtered)}")

# Приводим name и description к нижнему регистру, заполняем NaN
name_clean = df_filtered["name"].fillna("").str.lower()
desc_clean = df_filtered["description"].fillna("").str.lower()

# Паттерны для поиска C/C++
patterns_find = r"c\+\+|cpp|си\+\+|\bc\b|\bси\b"

# Исключения для описания (чтобы не поймать C#)
patterns_exclude_desc = r"c#|c-sharp|си шарп|1c|1с|css|c-level|objective-c|java"

# Исключения для названия (другие ЯП)
# Если в названии есть Java, Python, 1C и т.д. - это не C вакансия
patterns_exclude_name = r"java|python|1с|1c|php|ruby|golang|\bgo\b|react|angular|vue|scala|kotlin|swift|android|ios|верстальщик|frontend|фронтенд|javascript|js\b|typescript|ts\b|c#|c-sharp"

# Итоговая логика:
# (Ищем C++ в name ИЛИ description) 
# И (НЕ C# в description) 
# И (НЕ другие ЯП в name)
mask_find = name_clean.str.contains(patterns_find, regex=True) | desc_clean.str.contains(patterns_find, regex=True)
mask_not_sharp = ~desc_clean.str.contains(patterns_exclude_desc, regex=True)
mask_not_other_langs = ~name_clean.str.contains(patterns_exclude_name, regex=True)

mask_final = mask_find & mask_not_sharp & mask_not_other_langs

df_cpp = df_filtered[mask_final].copy()

print(f"\nНайдено вакансий C/C++: {len(df_cpp)}")

# Проверка результатов
print("\nТоп-12 названий найденных вакансий:")
print(df_cpp["name"].value_counts().head(12).to_string())

print("\nРаспределение по ролям:")
print(df_cpp["role"].value_counts().to_string())

Всего вакансий: 229388
Вакансий с нужными ролями: 93407

Найдено вакансий C/C++: 5653

Топ-12 названий найденных вакансий:
name
Инженер-программист                        202
Программист С++                            111
Программист C/С++                           85
Инженер-программист микроконтроллеров       78
Разработчик С++                             74
Программист микроконтроллеров               66
Разработчик C++                             61
Программист C++                             53
Ведущий инженер-программист                 48
Программист                                 42
Инженер-программист встраиваемых систем     41
C++ Developer                               38

Распределение по ролям:
role
Программист, разработчик          5377
Руководитель группы разработки     276


In [21]:
output_file = "cpp_vacancies.csv"
df_cpp.to_csv(output_file, sep="\t", index=False)
print(f"Данные сохранены в {output_file}")

Данные сохранены в cpp_vacancies.csv


In [4]:
top_regions = df_cpp["area"].value_counts().head(15)
print("\nТоп-15 регионов по количеству вакансий")
print(top_regions)


Топ-15 регионов по количеству вакансий
area
Москва             2632
Санкт-Петербург     957
Нижний Новгород     237
Новосибирск         189
Екатеринбург        164
Казань              120
Уфа                  86
Томск                83
Воронеж              60
Зеленоград           51
Пермь                50
Самара               50
Ростов-на-Дону       48
Краснодар            45
Челябинск            43
Name: count, dtype: int64
